In [1]:
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

path = "Pretrained/MiniLM"
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=4).to(device)


def load(name, shuffle):
    df = pd.read_csv("Datasets/" + name, header=None)
    enc = tokenizer(df[1].tolist(), truncation=True, padding="max_length", max_length=32, return_tensors="pt")
    labels = torch.tensor(df[0].values - 1)
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], labels)
    print(name, " Samples:", len(df))
    return DataLoader(ds, batch_size=64, shuffle=shuffle)


train_loader = load("ag_train.csv", True)
test_loader = load("ag_test.csv", False)

opt = torch.optim.AdamW(model.parameters(), lr=3e-5)

model.train()
step = 0
for ids, mask, labels in train_loader:
    opt.zero_grad()
    out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=labels.to(device))
    out.loss.backward()
    opt.step()
    if step % 300 == 0:
        print("Step", step, " Loss =", round(out.loss.item(), 4))
    step += 1

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for ids, mask, labels in test_loader:
        out = model(input_ids=ids.to(device), attention_mask=mask.to(device))
        pred = out.logits.argmax(dim=1).cpu()
        correct += (pred == labels).sum().item()
        total += len(labels)

print("Test Accuracy:", round(correct / total, 4))


C:\Users\Shubham\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Pretrained/MiniLM and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ag_train.csv  Samples: 120000
ag_test.csv  Samples: 7600
Step 0  Loss = 1.3898
Step 300  Loss = 0.3729
Step 600  Loss = 0.4985
Step 900  Loss = 0.494
Step 1200  Loss = 0.3358
Step 1500  Loss = 0.4195
Step 1800  Loss = 0.2857
Test Accuracy: 0.888
